In [4]:
# keras module for building LSTM 
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Sequential
import tensorflow.keras.utils as ku 
import tensorflow as tf

# set seeds for reproducability
import numpy as np
tf.random.set_seed(2)
np.random.seed(1)

import pandas as pd
import string, os 

import warnings
warnings.filterwarnings("ignore")
warnings.simplefilter(action='ignore', category=FutureWarning)


## 2. Load the dataset

Load the dataset of news headlines

In [6]:
curr_dir = './'
all_headlines = []
for filename in os.listdir(curr_dir):
    if 'Articles' in filename:
        article_df = pd.read_csv(curr_dir + filename)
        all_headlines.extend(list(article_df.headline.values))
        break

all_headlines = [h for h in all_headlines if h != "Unknown"]
len(all_headlines)

831

## 3. Dataset preparation

### 3.1 Dataset cleaning 

In dataset preparation step, we will first perform text cleaning of the data which includes removal of punctuations and lower casing all the words. 

In [7]:
def clean_text(txt):
    txt = "".join(v for v in txt if v not in string.punctuation).lower()
    txt = txt.encode("utf8").decode("ascii",'ignore')
    return txt 

corpus = [clean_text(x) for x in all_headlines]
corpus[:10]

['finding an expansive view  of a forgotten people in niger',
 'and now  the dreaded trump curse',
 'venezuelas descent into dictatorship',
 'stain permeates basketball blue blood',
 'taking things for granted',
 'the caged beast awakens',
 'an everunfolding story',
 'oreilly thrives as settlements add up',
 'mouse infestation',
 'divide in gop now threatens trump tax plan']

### 3.2 Generating Sequence of N-gram Tokens

In [8]:
tokenizer = Tokenizer()

def get_sequence_of_tokens(corpus):
    ## tokenization
    tokenizer.fit_on_texts(corpus)
    total_words = len(tokenizer.word_index) + 1
    
    ## convert data to sequence of tokens 
    input_sequences = []
    for line in corpus:
        token_list = tokenizer.texts_to_sequences([line])[0]
        for i in range(1, len(token_list)):
            n_gram_sequence = token_list[:i+1]
            input_sequences.append(n_gram_sequence)
    return input_sequences, total_words

inp_sequences, total_words = get_sequence_of_tokens(corpus)
inp_sequences[:10]

[[169, 17],
 [169, 17, 665],
 [169, 17, 665, 367],
 [169, 17, 665, 367, 4],
 [169, 17, 665, 367, 4, 2],
 [169, 17, 665, 367, 4, 2, 666],
 [169, 17, 665, 367, 4, 2, 666, 170],
 [169, 17, 665, 367, 4, 2, 666, 170, 5],
 [169, 17, 665, 367, 4, 2, 666, 170, 5, 667],
 [6, 80]]

In [9]:
def generate_padded_sequences(input_sequences):
    max_sequence_len = max([len(x) for x in input_sequences])
    input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))
    
    predictors, label = input_sequences[:,:-1],input_sequences[:,-1]
    label = ku.to_categorical(label, num_classes=total_words)
    return predictors, label, max_sequence_len

predictors, label, max_sequence_len = generate_padded_sequences(inp_sequences)

## 4. LSTMs for Text Generation


In [10]:
def create_model(max_sequence_len, total_words):
    input_len = max_sequence_len - 1
    model = Sequential()
    
    # Add Input Embedding Layer
    model.add(Embedding(total_words, 10, input_length=input_len))
    
    # Add Hidden Layer 1 - LSTM Layer
    model.add(LSTM(100))
    model.add(Dropout(0.1))
    
    # Add Output Layer
    model.add(Dense(total_words, activation='softmax'))

    model.compile(loss='categorical_crossentropy', optimizer='adam')
    
    return model

model = create_model(max_sequence_len, total_words)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [11]:
model.fit(predictors, label, epochs=100, verbose=1)


Epoch 1/100
151/151 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - loss: 7.3710
Epoch 2/100
151/151 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 6.9023
Epoch 3/100
151/151 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 6.8002
Epoch 4/100
151/151 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 6.7244
Epoch 5/100
151/151 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 6.6538
Epoch 6/100
151/151 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 6.5834
Epoch 7/100
151/151 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 6.4903
Epoch 8/100
151/151 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 6.3673
Epoch 9/100
151/151 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 6.2573
Epoch 10/100
151/151 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 6.1690
Epoch 11/100
151/151 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 6.0713
Epoch 12/100
151/151 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 5.9641
Epoch 13/100
151/151 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 5.8856
Epoch 14/100
151/151 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 5.8553
Epoch 15/100
151/151 ━━━━━━━━━━

## 5. Generating the text 

In [12]:
def generate_text(seed_text, next_words, model, max_sequence_len):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
        predicted = np.argmax(model.predict(token_list, verbose=0), axis=-1)[0]
        
        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break
        
        if output_word:
            seed_text += " " + output_word
        else:
            break
    return seed_text.title()


## 6. Some Results

In [13]:
print (generate_text("united states", 5, model, max_sequence_len))
print (generate_text("preident trump", 4, model, max_sequence_len))
print (generate_text("donald trump", 4, model, max_sequence_len))
print (generate_text("india and china", 4, model, max_sequence_len))
print (generate_text("new york", 4, model, max_sequence_len))
print (generate_text("science and technology", 5, model, max_sequence_len))

United States House To Elevating Police Judge
Preident Trump Find A Behemoths Swagger
Donald Trump But A Small Person
India And China Became Ms Intelligence Aims
New York Today Passover On The
Science And Technology May Ok To Try Up
